# Canonical structural-pruning pilot (seed 101)

This notebook is manually launched by the researcher. It clones an immutable Git commit, runs only the approved pilot, and writes independent condition outputs under `/kaggle/working`. Historical results are never loaded or pooled.

In [ ]:
import hashlib
import json
import shutil
from pathlib import Path

REPO_URL = "https://github.com/r1zons/prunable_gnn_codex.git"
PINNED_COMMIT = "2846da87a12428f21d2d751c6e079d654b68c61d"
SELECTED_CONDITIONS = "all"  # Or a list containing one or more documented condition names.
REPO_ROOT = Path("/kaggle/working/prunable_gnn_codex")
OUTPUT_ROOT = Path("/kaggle/working/prunable_gnn_kaggle_pilot")
ARCHIVE_BASE = Path("/kaggle/working/prunable_gnn_kaggle_pilot_results")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def archive_partial_results():
    archive_path = Path(shutil.make_archive(str(ARCHIVE_BASE), "zip", root_dir=OUTPUT_ROOT))
    digest_state = hashlib.sha256()
    with archive_path.open("rb") as archive_handle:
        for chunk in iter(lambda: archive_handle.read(1024 * 1024), b""):
            digest_state.update(chunk)
    digest = digest_state.hexdigest()
    Path(f"{archive_path}.sha256").write_text(f"{digest}  {archive_path.name}\n", encoding="utf-8")
    print(f"Output archive: {archive_path}")
    print(f"Archive SHA-256: {digest}")
    return archive_path

In [ ]:
import re
import subprocess

bootstrap_log = OUTPUT_ROOT / "bootstrap.log"
try:
    if not re.fullmatch(r"[0-9a-f]{40}", PINNED_COMMIT):
        raise RuntimeError("Set PINNED_COMMIT to the full pushed implementation commit SHA before Save & Run All.")
    if REPO_ROOT.exists():
        raise RuntimeError(f"Refusing to reuse existing checkout: {REPO_ROOT}")
    with bootstrap_log.open("w", encoding="utf-8") as log:
        subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_ROOT)], stdout=log, stderr=subprocess.STDOUT, check=True, text=True)
        subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT], cwd=REPO_ROOT, stdout=log, stderr=subprocess.STDOUT, check=True, text=True)
    actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, text=True).strip()
    if actual_commit != PINNED_COMMIT:
        raise RuntimeError(f"Commit verification failed: expected {PINNED_COMMIT}, found {actual_commit}")
    print(f"Pinned checkout verified: {actual_commit}")
except Exception as exc:
    (OUTPUT_ROOT / "bootstrap_failure.json").write_text(json.dumps({"error_type": type(exc).__name__, "error": str(exc)}, indent=2), encoding="utf-8")
    archive_partial_results()
    raise

In [ ]:
import os
import sys

install_log = OUTPUT_ROOT / "dependency_install.log"
try:
    try:
        import torch
    except ImportError as exc:
        raise RuntimeError("The Kaggle image must provide PyTorch; the pilot will not replace its CUDA build.") from exc
    with install_log.open("w", encoding="utf-8") as log:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "kaggle/requirements-pilot.txt")],
            stdout=log, stderr=subprocess.STDOUT, check=True, text=True,
        )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO_ROOT)],
            stdout=log, stderr=subprocess.STDOUT, check=True, text=True,
        )
    freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
    (OUTPUT_ROOT / "pip_freeze.txt").write_text(freeze, encoding="utf-8")
    import torch_geometric
    print(json.dumps({"torch": torch.__version__, "torch_geometric": torch_geometric.__version__, "cuda": torch.cuda.is_available()}, indent=2))
except Exception as exc:
    (OUTPUT_ROOT / "dependency_failure.json").write_text(json.dumps({"error_type": type(exc).__name__, "error": str(exc)}, indent=2), encoding="utf-8")
    archive_partial_results()
    raise

In [ ]:
runner = [
    sys.executable, "-u", str(REPO_ROOT / "scripts/run_kaggle_pilot.py"),
    "--repo-root", str(REPO_ROOT),
    "--expected-commit", PINNED_COMMIT,
]
if SELECTED_CONDITIONS == "all":
    runner.append("--all")
else:
    if not isinstance(SELECTED_CONDITIONS, list) or not SELECTED_CONDITIONS:
        raise RuntimeError("SELECTED_CONDITIONS must be 'all' or a non-empty list.")
    for condition in SELECTED_CONDITIONS:
        runner.extend(["--condition", str(condition)])

run_error = None
runner_log = OUTPUT_ROOT / "pilot_runner.log"
try:
    with runner_log.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(runner, cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        return_code = process.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, runner)
except Exception as exc:
    run_error = exc
    (OUTPUT_ROOT / "notebook_runner_failure.json").write_text(json.dumps({"error_type": type(exc).__name__, "error": str(exc)}, indent=2), encoding="utf-8")
finally:
    archive_partial_results()
if run_error is not None:
    raise run_error
print("Seed-101 pilot completed. Review validation summaries before authorizing seeds 202 and 303.")